# Multi-Agent · Day 32 — **CrewAI Deep Dive**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~2.5 hrs

CrewAI's pitch is role-play: give each agent a **role**, a **goal** and a **backstory**, hand the crew a list of **tasks**, and let outputs flow from one to the next. It's the message-passing end of yesterday's spectrum, and it's the fastest way to stand up a working crew — which is exactly why you need to know where it stops being the right tool.

Today:
1. CrewAI's object model — `Agent`, `Task`, `Crew`, `Process`.
2. **Build the framework** in ~50 lines, so nothing is magic (and it runs keyless).
3. A 3-agent crew — **researcher → analyst → writer** on *"Analyse Barclays Q4 2024 results"*.
4. **Delegation** and hierarchical process.
5. **JSON audit trail** + when to choose CrewAI over LangGraph.

Real CrewAI needs `pip install crewai` and an LLM key. We rebuild the core with a deterministic stub LLM so the *orchestration* is what you study — then you'll read the real API and recognise every piece.

## 1. The object model

Four concepts. Everything CrewAI does is these four plus an LLM.

In [1]:
MODEL = [
    ("Agent",   "role + goal + backstory + tools + allow_delegation",
                "a persona — the system prompt, essentially"),
    ("Task",    "description + expected_output + agent + context",
                "one unit of work; its output feeds the next task"),
    ("Crew",    "agents + tasks + process + verbose",
                "the runner; owns execution order"),
    ("Process", "sequential | hierarchical",
                "sequential = chain; hierarchical = a manager LLM delegates"),
]
for name, fields, meaning in MODEL:
    print(f"{name:9} {fields}\n{'':9} -> {meaning}\n")

print("""Real CrewAI (for reference — needs `pip install crewai` + an LLM key):

    from crewai import Agent, Task, Crew, Process

    researcher = Agent(role="Research Analyst", goal="Find the numbers behind the headline",
                       backstory="15 years covering UK banks.", tools=[search_tool], verbose=True)
    task = Task(description="Analyse Barclays Q4 2024 results.",
                expected_output="5 bullets with figures", agent=researcher)
    crew = Crew(agents=[researcher], tasks=[task], process=Process.sequential)
    result = crew.kickoff()
""")

Agent     role + goal + backstory + tools + allow_delegation
          -> a persona — the system prompt, essentially

Task      description + expected_output + agent + context
          -> one unit of work; its output feeds the next task

Crew      agents + tasks + process + verbose
          -> the runner; owns execution order

Process   sequential | hierarchical
          -> sequential = chain; hierarchical = a manager LLM delegates

Real CrewAI (for reference — needs `pip install crewai` + an LLM key):

    from crewai import Agent, Task, Crew, Process

    researcher = Agent(role="Research Analyst", goal="Find the numbers behind the headline",
                       backstory="15 years covering UK banks.", tools=[search_tool], verbose=True)
    task = Task(description="Analyse Barclays Q4 2024 results.",
                expected_output="5 bullets with figures", agent=researcher)
    crew = Crew(agents=[researcher], tasks=[task], process=Process.sequential)
    result = crew.kickoff

☝ Notice what `Agent` really is: `role`, `goal` and `backstory` are concatenated into a system prompt. That's the whole "role-play" mechanism — no special model, no fine-tune. It matters because it tells you where CrewAI's quality comes from (**prompting**) and where its failures come from (an agent with a vague goal produces vague output, and nothing in the framework will catch that — which is why yesterday's `__post_init__` validation earns its place).

## 2. CrewAI in 50 lines

Rebuild the core: `Agent` and `Task` (Day 32's Python dataclasses, reused), a stub LLM, and a `Crew.kickoff()` that chains outputs. Deterministic and keyless.

In [2]:
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Callable

@dataclass(frozen=True, slots=True)
class Agent:
    role: str
    goal: str
    backstory: str = ""
    allow_delegation: bool = False

    def system_prompt(self) -> str:
        return f"You are {self.role}. Goal: {self.goal}. Backstory: {self.backstory}"

@dataclass
class Task:
    description: str
    expected_output: str
    agent: Agent
    context: list["Task"] = field(default_factory=list)   # outputs fed in as context
    output: str | None = None

    def __post_init__(self) -> None:
        if len(self.description) < 15:
            raise ValueError(f"Task description too vague: {self.description!r}")

def stub_llm(system: str, user: str) -> str:
    """Deterministic stand-in for an LLM call — keyed on the agent's role."""
    role = system.split(".")[0].removeprefix("You are ").strip()
    canned = {
        "Research Analyst": ("FINDINGS\n  • FY24 group income £26.8bn (+6% YoY)\n"
                             "  • Q4 NII £3.1bn, UK NIM 3.11%\n  • Impairments £0.4bn in Q4, below guidance\n"
                             "  • CET1 ratio 13.6%\n  • Costs £16.7bn, cost:income 62%"),
        "Financial Analyst": ("ANALYSIS\n  • Income growth is NII-led -> rate-sensitive, not structural\n"
                              "  • Cost:income 62% vs 60% target -> efficiency gap persists\n"
                              "  • Low impairments + CET1 13.6% -> capacity for buybacks\n"
                              "  RISK: NII tailwind reverses when rates fall"),
        "Report Writer": ("BARCLAYS Q4 2024 — ONE-PAGE BRIEF\n"
                          "  Income £26.8bn (+6%), NII-led. Cost:income 62% (target 60%).\n"
                          "  Impairments benign at £0.4bn; CET1 13.6% supports distributions.\n"
                          "  Watch: rate cuts unwind the NII tailwind; efficiency gap is the swing factor."),
    }
    body = canned.get(role, f"[{role}] no canned output")
    return body + (f"\n  (used context: {len(user.split('### CONTEXT'))-1} upstream output(s))")

@dataclass
class Crew:
    agents: list[Agent]
    tasks: list[Task]
    process: str = "sequential"
    llm: Callable[[str, str], str] = stub_llm
    log: list[dict] = field(default_factory=list)

    def kickoff(self, verbose: bool = True) -> str:
        for task in self.tasks:
            ctx = "".join(f"\n### CONTEXT from {c.agent.role}:\n{c.output}" for c in task.context)
            user = f"TASK: {task.description}\nEXPECTED OUTPUT: {task.expected_output}{ctx}"
            task.output = self.llm(task.agent.system_prompt(), user)
            self.log.append({"agent": task.agent.role, "task": task.description,
                             "context_from": [c.agent.role for c in task.context],
                             "at": datetime.now(timezone.utc).isoformat()})
            if verbose:
                print(f"\n{'='*70}\n[{task.agent.role}]\n{task.output}")
        return self.tasks[-1].output

print("crew engine ready — no API key needed")

crew engine ready — no API key needed


☝ Fifty lines, and the essential mechanic is visible: **`kickoff` loops the tasks and stuffs each task's `context` outputs into the next prompt.** That's it. Real CrewAI adds tool calling, memory, retries and a proper LLM — but the orchestration is this loop. Once you've seen it, "the crew produced nonsense" stops being mysterious: it means a prompt got a bad context, and you can point at the exact concatenation that did it.

## 3. The crew — researcher → analyst → writer

In [3]:
researcher = Agent(
    role="Research Analyst",
    goal="Find the hard numbers behind Barclays' Q4 2024 headline results",
    backstory="15 years covering UK banks; you never state a figure without a source.",
)
analyst = Agent(
    role="Financial Analyst",
    goal="Turn raw figures into the two or three things that actually matter",
    backstory="Ex-buyside; you distrust headline growth and look for what drives it.",
    allow_delegation=True,
)
writer = Agent(
    role="Report Writer",
    goal="Produce a one-page brief a busy MD reads in 60 seconds",
    backstory="You write for executives: no jargon, no hedging, figures in every claim.",
)

t1 = Task("Analyse Barclays Q4 2024 results and extract the key revenue drivers.",
          "5 bullets, each with a figure", researcher)
t2 = Task("Interpret the findings: what is structural, what is a rate tailwind, what is the risk?",
          "3 bullets + 1 risk line", analyst, context=[t1])
t3 = Task("Write the one-page executive brief from the analysis.",
          "<=6 lines, plain English, figures included", writer, context=[t1, t2])

crew = Crew(agents=[researcher, analyst, writer], tasks=[t1, t2, t3])
final = crew.kickoff()


[Research Analyst]
FINDINGS
  • FY24 group income £26.8bn (+6% YoY)
  • Q4 NII £3.1bn, UK NIM 3.11%
  • Impairments £0.4bn in Q4, below guidance
  • CET1 ratio 13.6%
  • Costs £16.7bn, cost:income 62%
  (used context: 0 upstream output(s))

[Financial Analyst]
ANALYSIS
  • Income growth is NII-led -> rate-sensitive, not structural
  • Cost:income 62% vs 60% target -> efficiency gap persists
  • Low impairments + CET1 13.6% -> capacity for buybacks
  RISK: NII tailwind reverses when rates fall
  (used context: 1 upstream output(s))

[Report Writer]
BARCLAYS Q4 2024 — ONE-PAGE BRIEF
  Income £26.8bn (+6%), NII-led. Cost:income 62% (target 60%).
  Impairments benign at £0.4bn; CET1 13.6% supports distributions.
  Watch: rate cuts unwind the NII tailwind; efficiency gap is the swing factor.
  (used context: 2 upstream output(s))


☝ Look at the `(used context: N upstream output(s))` line the stub prints — that's the token bill. The writer's prompt carries **both** upstream outputs because `context=[t1, t2]`; by task 3 you're re-sending most of the crew's work into one call. This is message passing's cost curve: cheap per hop, but `context` lists quietly recreate shared state if you pass everything to everyone.

The discipline: pass the **minimum** context each task needs. If the writer only needs the analysis, `context=[t2]` halves the prompt.

## 4. Delegation and the hierarchical process

`allow_delegation=True` lets an agent hand a sub-question to a colleague mid-task. `Process.hierarchical` puts a *manager* LLM in front that decides who does what — CrewAI's version of yesterday's supervisor topology.

In [4]:
def delegate(from_agent: Agent, to_agent: Agent, question: str, crew: Crew) -> str:
    """Mid-task handoff — what CrewAI's 'Delegate work to coworker' tool does."""
    if not from_agent.allow_delegation:
        raise PermissionError(f"{from_agent.role} is not permitted to delegate")
    crew.log.append({"agent": from_agent.role, "delegated_to": to_agent.role, "question": question,
                     "at": datetime.now(timezone.utc).isoformat()})
    answer = crew.llm(to_agent.system_prompt(), f"COLLEAGUE QUESTION from {from_agent.role}: {question}")
    print(f"[{from_agent.role}] --delegates--> [{to_agent.role}]: {question}")
    return answer

_ = delegate(analyst, researcher, "What was the Q4 impairment charge exactly?", crew)

try:
    delegate(writer, researcher, "Can you re-check CET1?", crew)
except PermissionError as exc:
    print("blocked ->", exc)

class HierarchicalCrew(Crew):
    """Process.hierarchical: a manager routes each task instead of a fixed chain."""

    def kickoff(self, verbose: bool = True) -> str:
        manager = Agent("Crew Manager", "Route each task to the best-suited agent")
        for task in self.tasks:
            chosen = self._route(task)
            print(f"[manager] '{task.description[:42]}...' -> {chosen.role}")
            task.agent = chosen
        return super().kickoff(verbose=False)

    def _route(self, task: Task) -> Agent:
        # a real manager asks the LLM; keyword routing keeps this deterministic
        d = task.description.lower()
        for kw, role in [("extract", "Research Analyst"), ("interpret", "Financial Analyst"),
                         ("write", "Report Writer")]:
            if kw in d:
                return next(a for a in self.agents if a.role == role)
        return self.agents[0]

print()
h = HierarchicalCrew(agents=[researcher, analyst, writer],
                     tasks=[Task("Extract the Q4 headline figures.", "5 bullets", researcher),
                            Task("Interpret what drives the income growth.", "3 bullets", researcher),
                            Task("Write the executive brief for the MD.", "6 lines", researcher)])
_ = h.kickoff()

[Financial Analyst] --delegates--> [Research Analyst]: What was the Q4 impairment charge exactly?
blocked -> Report Writer is not permitted to delegate

[manager] 'Extract the Q4 headline figures....' -> Research Analyst
[manager] 'Interpret what drives the income growth....' -> Financial Analyst
[manager] 'Write the executive brief for the MD....' -> Report Writer


☝ Two things worth carrying into a design review. `allow_delegation` is an **access boundary**, not a convenience flag — the writer being unable to delegate is the same control as a compliance agent being unable to write to the ledger. Set it `False` by default and turn it on deliberately.

And hierarchical process costs a **manager LLM call per task** on top of the work itself. It earns that when task→agent assignment genuinely can't be known upfront. If you can write the chain by hand — and for a research→analyse→write pipeline you can — sequential is cheaper, faster and fully deterministic. Choosing sequential here is the senior call.

## 5. The audit trail

In [5]:
import json

audit = {
    "crew": "barclays_q4_research",
    "process": crew.process,
    "agents": [asdict(a) for a in crew.agents],
    "steps": crew.log,
    "final_output": final,
}
print(json.dumps(audit, indent=2, default=str)[:900], "...")
print("\nsteps recorded:", len(crew.log))
print("delegations   :", [s for s in crew.log if "delegated_to" in s])

{
  "crew": "barclays_q4_research",
  "process": "sequential",
  "agents": [
    {
      "role": "Research Analyst",
      "goal": "Find the hard numbers behind Barclays' Q4 2024 headline results",
      "backstory": "15 years covering UK banks; you never state a figure without a source.",
      "allow_delegation": false
    },
    {
      "role": "Financial Analyst",
      "goal": "Turn raw figures into the two or three things that actually matter",
      "backstory": "Ex-buyside; you distrust headline growth and look for what drives it.",
      "allow_delegation": true
    },
    {
      "role": "Report Writer",
      "goal": "Produce a one-page brief a busy MD reads in 60 seconds",
      "backstory": "You write for executives: no jargon, no hedging, figures in every claim.",
      "allow_delegation": false
    }
  ],
  "steps": [
    {
      "agent": "Research Analyst",
      "task":  ...

steps recorded: 4
delegations   : [{'agent': 'Financial Analyst', 'delegated_to': 'Research An

☝ Every step is a record: which agent, which task, which upstream context, at what time. That JSON is what you hand an auditor asking *"why did the model say Barclays has capacity for buybacks?"* — you can walk it back to the figures the researcher produced.

Build this from day one. Retrofitting an audit trail onto a crew that's already in production means re-running everything, and in a bank the trail is often the *reason* the system is allowed to exist.

## 6. Recap — CrewAI is a prompt chain with good ergonomics

| CrewAI concept | What it really is | Watch out for |
|---|---|---|
| `Agent(role, goal, backstory)` | a system prompt | vague goal = vague output; nothing validates it |
| `Task(description, expected_output)` | a user prompt + output contract | `expected_output` is a *hint*, not a schema — validate with Pydantic |
| `context=[t1, t2]` | prompt concatenation | passing everything to everyone rebuilds shared state, at cost |
| `Process.sequential` | a for-loop | usually the right answer |
| `Process.hierarchical` | manager LLM routes | +1 LLM call per task |
| `allow_delegation` | an access boundary | default it to `False` |

**Choose CrewAI when** the work is a chain of role-shaped steps, you want a crew running this afternoon, and each step only needs its predecessor's output. **Choose LangGraph when** you need cycles, conditional routing, human-in-the-loop interrupts, checkpointing/resume, or a real audit of state at each step — i.e. most regulated banking flows.

They compose, too: a LangGraph node can `kickoff()` a crew. Tomorrow you build the LangGraph supervisor and get to compare the two honestly.